In [20]:
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

In [21]:
df = pd.read_csv('data.csv')
df = df.sort_values(by=['Year', 'Share'], ascending=[True, False])
df = df[df['G'] > 40]
df = df[df['MPPG'] >= 30]
df = df[df['FGA'] >= 2]
df = df[df['PTSPG'] >= 13]
df = df[df['TRBPG'] >= 3]
df = df[df['FG%'] >= 0.37]
df = df[df['FGAPG'] >= 10]
df = df[df['ASTPG'] >= 1]
df = df[df['PER'] >= 18]

In [22]:
# Experiment Cell
pd.concat([df[df['Year'] == 2004], df[df['Year'] == 2005]])
features = df.columns.to_list()
features = [feature for feature in df.columns.to_list() if feature not in ['Player', 'Share', 'Year']]
df[features]
df[df['Year'] == 2024]

,Player,G,GS,MP,FG,FGA,FG%,3P,3PA,3P%,...,OWS,DWS,WS,WS/48,OBPM,DBPM,BPM,VORP,Share,Year
9780,Nikola Jokić,79.0,79.0,2737.0,822.0,1411.0,0.583,83.0,231.0,0.359,...,12.0,5.1,17.0,0.299,9.0,4.2,13.2,10.6,0.935,2024
9777,Shai Gilgeous-Alexander,75.0,75.0,2553.0,796.0,1487.0,0.535,95.0,269.0,0.353,...,10.5,4.2,14.6,0.275,6.7,2.3,9.0,7.1,0.646,2024
9776,Luka Dončić,70.0,70.0,2624.0,804.0,1652.0,0.487,284.0,744.0,0.382,...,8.5,3.5,12.0,0.220,8.3,1.7,9.9,8.0,0.572,2024
9778,Giannis Antetokounmpo,73.0,73.0,2567.0,837.0,1369.0,0.611,34.0,124.0,0.274,...,9.5,3.7,13.2,0.246,6.7,2.4,9.0,7.2,0.194,2024
9779,Jalen Brunson,77.0,77.0,2726.0,790.0,1648.0,0.479,211.0,526.0,0.401,...,8.8,2.4,11.2,0.198,6.3,-0.4,5.8,5.4,0.143,2024
9783,Jayson Tatum,74.0,74.0,2645.0,672.0,1426.0,0.471,229.0,609.0,0.376,...,6.4,4.1,10.4,0.189,4.5,0.6,5.1,4.7,0.087,2024
9781,Anthony Edwards,79.0,78.0,2770.0,718.0,1558.0,0.461,190.0,532.0,0.357,...,2.9,4.7,7.5,0.130,2.7,0.5,3.3,3.7,0.018,2024
9801,Domantas Sabonis,82.0,82.0,2928.0,634.0,1068.0,0.594,33.0,87.0,0.379,...,8.6,4.0,12.6,0.206,4.0,2.4,6.5,6.2,0.003,2024
9782,Kevin Durant,75.0,75.0,2791.0,751.0,1436.0,0.523,168.0,407.0,0.413,...,5.1,3.2,8.3,0.142,4.0,0.1,4.0,4.3,0.001,2024
9784,De'Aaron Fox,74.0,74.0,2659.0,720.0,1549.0,0.465,214.0,580.0,0.369,...,3.3,3.2,6.5,0.117,2.6,0.1,2.7,3.2,0.000,2024


In [23]:
def df_concat_stats(df, startYear, endYear):
    final_df = pd.DataFrame()
    if startYear == endYear:
        return df
    while startYear <= endYear:
        if final_df.empty:
            final_df = df[df['Year'] == startYear]
        else:
            final_df = pd.concat([final_df, df[df['Year'] == startYear]])
        startYear +=1
    return final_df

# TESTS #
test = df_concat_stats(df, 2004, 2004)
test.iloc[0:5]['Player'].tolist()

['Kevin Garnett',
 'Tim Duncan',
 "Jermaine O'Neal",
 'Peja Stojaković',
 'Kobe Bryant']

In [26]:
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
years = range(2005, 2025)

results = []
features = [feature for feature in df.columns.to_list() if feature not in ['Player', 'Share', 'Year']]


right = 0
wrong = 0

for year in years:
    model = XGBRegressor(n_estimators=16, max_depth=5, learning_rate = 0.2745, subsample=1, colsample_bytree=1)

    train_df = df_concat_stats(df, 2004, year-1)
    test_df = df[df['Year'] == year]
    
    X_train = train_df[features]
    y_train = train_df['Share']
    
    X_test = test_df[features]
    y_test = test_df['Share']

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_pred, y_test)
    
    test_copy = test_df.copy()
    test_copy['Predicted Share'] = y_pred
    actual_mvp = test_copy.sort_values('Share', ascending=False).iloc[0]['Player']
    predicted_mvp = test_copy.sort_values('Predicted Share', ascending=False).iloc[0]['Player']
    top_five = test_copy.sort_values('Predicted Share', ascending=False).iloc[0:5]['Player'].to_list()
    
    
    results.append({
        "Train Years": [2004, year-1],
        "Test Year": year,
        "MSE": mse,
        "Predicted MVP": predicted_mvp,
        "Actual MVP": actual_mvp,
        "Top Five": top_five
    })

    
    if predicted_mvp == actual_mvp:
        right += 1
    else:
        wrong += 1

print("Right: " + str(right) + ", Wrong: " + str(wrong))
results


Right: 11, Wrong: 9


[{'Train Years': [2004, 2004],
  'Test Year': 2005,
  'MSE': 0.0008727087704868736,
  'Predicted MVP': 'Steve Nash',
  'Actual MVP': 'Steve Nash',
  'Top Five': ['Steve Nash',
   "Shaquille O'Neal",
   'Dirk Nowitzki',
   'Allen Iverson',
   'Tim Duncan']},
 {'Train Years': [2004, 2005],
  'Test Year': 2006,
  'MSE': 0.036800013532978654,
  'Predicted MVP': 'Steve Nash',
  'Actual MVP': 'Steve Nash',
  'Top Five': ['Steve Nash',
   'Yao Ming',
   'Shawn Marion',
   'Allen Iverson',
   'Carmelo Anthony']},
 {'Train Years': [2004, 2006],
  'Test Year': 2007,
  'MSE': 0.017421988579660916,
  'Predicted MVP': 'Tim Duncan',
  'Actual MVP': 'Dirk Nowitzki',
  'Top Five': ['Tim Duncan',
   'Steve Nash',
   'Dirk Nowitzki',
   'Chauncey Billups',
   'LeBron James']},
 {'Train Years': [2004, 2007],
  'Test Year': 2008,
  'MSE': 0.025929126593176043,
  'Predicted MVP': 'Chris Paul',
  'Actual MVP': 'Kobe Bryant',
  'Top Five': ['Chris Paul',
   "Amar'e Stoudemire",
   'LeBron James',
   'Manu Gi